# Tabular prediction with SAP-RPT-1.5 and SAP-RPT-1.6 using direct REST calls

**SAP-RPT** is SAP's foundation model for tables. You send a table in which some cells are unknown; the model looks at the complete rows of the same table and fills in the unknown cells. There is no training step: the examples travel with each request.

```
              PRODUCT          PRICE    COSTCENTER            DISCOUNT_RATE
context  ->   Office Chair     150.80   Office Furniture      0.08
rows          Server Rack     2200.00   Data Infrastructure   0.15
              ...
query    ->   Couch            999.99   [PREDICT]             [PREDICT]
rows          UPS Battery      450.00   [PREDICT]             [PREDICT]
```

* **Context rows** are complete examples: rows where you already know the answer.
* **Query rows** contain the placeholder `[PREDICT]` in the cells you want filled in.
* A **classification** target is a category (here `COSTCENTER`): the model picks one of the values it has seen in the context.
* A **regression** target is a number (here `DISCOUNT_RATE`): the model estimates a value.

**Why direct REST?** The native SAP AI SDK (`../RPT-1_native_sdk.ipynb`) only knows the fields of the first RPT-1 version. The newer SAP-RPT-1.5 and 1.6 models add fields such as explanations, value ranges for numbers and `context_mode`, which the SDK drops. Plain HTTP requests keep every field.

What this notebook covers:

* The anatomy of one REST call (section 2).
* A request that fills a category and a number at the same time, with ranked alternatives and value ranges (sections 3 to 5).
* Explanations: which columns and which example rows the model relied on (section 6).
* A fair comparison of the four model variants on a real dataset with known answers (section 7).
* Compressing large requests (section 8).

**Prerequisites:** running SAP AI Core deployments of `sap-rpt-1.5`, `sap-rpt-1.5-large`, `sap-rpt-1.6` and `sap-rpt-1.6-large` (scenario `foundation-models`, executable `aicore-sap`), and the `AICORE_*` values of your service key in a `.env` file (see `.env.example`).

## 1. Setup

`rpt_client_api.py` (next to this notebook) is a small client. It reads the `AICORE_*` variables, gets and renews the access token, and finds the running deployment for a model name, so no deployment ID has to be written into the code.

In [10]:
import json
import os
import time

import numpy as np
import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from tqdm.auto import tqdm

from rpt_client_api import RPTClientAPI, flatten_predictions, map_explanation_rows

# Finds rest/.env or, walking up, RPT-1/python/.env.
_ = load_dotenv(find_dotenv(usecwd=True))
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

## 2. Anatomy of one REST call

Everything an RPT call needs, written with the `requests` library only:

| Step | Request | Purpose |
|---|---|---|
| 1. Access token | `POST {AICORE_AUTH_URL}/oauth/token` with `grant_type=client_credentials`, client ID and secret | proves who you are; valid for a limited time |
| 2. Deployment URL | `{AICORE_BASE_URL}/v2/inference/deployments/{deployment_id}` | the address of your running model |
| 3. Prediction | `POST {deployment_url}/predict` with headers `Authorization: Bearer <token>` and `AI-Resource-Group: <group>` | sends the table, returns the filled cells |

`RPTClientAPI` looks up step 2 for you (it searches the AI Core configurations for the model name and picks the matching running deployment). Deployment URLs and tokens are not printed here: keep them out of notebooks and logs.

The table below is tiny on purpose; it only shows the mechanics.

In [11]:
token = requests.post(
    os.environ["AICORE_AUTH_URL"].rstrip("/") + "/oauth/token",
    data={
        "grant_type": "client_credentials",
        "client_id": os.environ["AICORE_CLIENT_ID"],
        "client_secret": os.environ["AICORE_CLIENT_SECRET"],
    },
    timeout=60,
).json()["access_token"]

deployment_url = RPTClientAPI(model_name="sap-rpt-1.6").deployment_url

minimal_body = {
    "index_column": "ID",
    "prediction_config": {
        "target_columns": [
            {"name": "COSTCENTER", "prediction_placeholder": "[PREDICT]", "task_type": "classification"}
        ]
    },
    "rows": [
        {"ID": "1", "PRODUCT": "Office Chair", "COSTCENTER": "Office Furniture"},
        {"ID": "2", "PRODUCT": "Server Rack", "COSTCENTER": "Data Infrastructure"},
        {"ID": "3", "PRODUCT": "Standing Desk", "COSTCENTER": "[PREDICT]"},
    ],
}
raw = requests.post(
    deployment_url + "/predict",
    headers={
        "Authorization": f"Bearer {token}",
        "AI-Resource-Group": os.environ["AICORE_RESOURCE_GROUP"],
        "Content-Type": "application/json",
    },
    json=minimal_body,
    timeout=120,
)
print("HTTP", raw.status_code)
print(json.dumps({k: v for k, v in raw.json().items() if k != "id"}, indent=2))

HTTP 200
{
  "explanations": null,
  "metadata": {
    "context_mode": "default",
    "num_columns": 3,
    "num_predictions": 1,
    "num_query_rows": 1,
    "num_rows": 3
  },
  "predictions": [
    {
      "COSTCENTER": [
        {
          "confidence": 0.92,
          "confidence_interval": null,
          "prediction": "Data Infrastructure"
        }
      ],
      "ID": "3"
    }
  ],
  "status": {
    "code": 0,
    "message": "ok"
  }
}


With only two example rows the answer carries no real meaning (a desk is not data infrastructure). The model needs representative examples, as in the next sections.

**Two status checks.** A call can fail in two places, and both must be checked:

* the **HTTP status**: transport and request format; a badly formed body returns 422;
* the **`status.code`** inside the response: `0` ok, `1` warning, `2` invalid input, `3` server error.

`RPTClientAPI.predict_json` checks both and raises `RPTAPIError` with the server's message, so the rest of the notebook uses it.

## 3. The sample data

`../sample_orders.csv` is a small order table shared with the other RPT examples in this repository. Each row is one order:

| Column | Meaning |
|---|---|
| `ID` | order number |
| `PRODUCT`, `PRICE`, `ORDERDATE`, `QUANTITY` | what was ordered, unit price, date, number of units |
| `COSTCENTER` | the internal cost center that pays for it (a category) |
| `DISCOUNT_RATE` | the discount granted, as a share of the price (a number) |

We hide `COSTCENTER` and `DISCOUNT_RATE` for two orders and let the model fill them in. The true values are kept aside to check the answers.

In [12]:
orders = pd.read_csv("../sample_orders.csv", dtype={"ID": str})
QUERY_IDS = ["35", "711"]
TARGETS = ["COSTCENTER", "DISCOUNT_RATE"]

truth = orders.loc[orders["ID"].isin(QUERY_IDS), ["ID", *TARGETS]].set_index("ID")
table = orders.copy()
# Targets become text columns so they can hold the "[PREDICT]" placeholder.
table[TARGETS] = table[TARGETS].astype(object)
table.loc[table["ID"].isin(QUERY_IDS), TARGETS] = "[PREDICT]"
table

,ID,PRODUCT,PRICE,ORDERDATE,QUANTITY,COSTCENTER,DISCOUNT_RATE
0,44,Office Chair,150.80,2025-11-02,12,Office Furniture,0.08
1,104,Server Rack,2200.00,2025-11-01,2,Data Infrastructure,0.15
2,88,Laptop Stand,49.90,2025-11-03,25,Office Furniture,0.05
3,205,Standing Desk,640.00,2025-11-05,6,Office Furniture,0.1
4,306,Monitor 27 inch,289.99,2025-11-08,10,Office Furniture,0.07
5,401,Network Switch,980.00,2025-11-10,3,Data Infrastructure,0.12
6,502,Keyboard Mechanical,120.00,2025-11-12,30,Office Furniture,0.06
7,35,Couch,999.99,2025-11-28,4,[PREDICT],[PREDICT]
8,610,Cable Management Kit,25.50,2025-11-15,50,Office Furniture,0.03
9,711,UPS Battery Backup,450.00,2025-11-18,5,[PREDICT],[PREDICT]


## 4. Build the request

The request is a JSON object. Its parts:

| Field | Meaning |
|---|---|
| `rows` | the whole table, context rows and query rows together, as a list of `{column: value}` objects |
| `index_column` | a column that identifies each row (here `ID`), so each answer can be matched back to its row |
| `prediction_config.target_columns` | one entry per column to fill in |
| &nbsp;&nbsp;`name` | the column name |
| &nbsp;&nbsp;`prediction_placeholder` | the marker of the cells to fill (`[PREDICT]`) |
| &nbsp;&nbsp;`task_type` | `classification` (pick a category) or `regression` (estimate a number). The model can guess this, but setting it avoids surprises. |
| &nbsp;&nbsp;`top_k` | for categories: how many ranked alternatives to return (here 2: best and second-best) |
| `prediction_config.explanations` | optional: also return which columns and example rows mattered most (section 6) |
| `data_schema` | optional: the type of each column, `string`, `numeric` or `date`, so for example `ORDERDATE` is read as a date. Dates must be written `YYYY-MM-DD`. |

> **Several targets in one request are predicted together.** The model fills `COSTCENTER` and `DISCOUNT_RATE` jointly, so the answers can differ from two separate requests with one target each.

In [13]:
payload = {
    "index_column": "ID",
    "prediction_config": {
        "target_columns": [
            {"name": "COSTCENTER", "prediction_placeholder": "[PREDICT]", "task_type": "classification", "top_k": 2},
            {"name": "DISCOUNT_RATE", "prediction_placeholder": "[PREDICT]", "task_type": "regression"},
        ],
        "explanations": {"top_column_scores": 3, "top_relevant_context_rows": 2},
    },
    "data_schema": {
        "ID": {"dtype": "string"},
        "PRODUCT": {"dtype": "string"},
        "PRICE": {"dtype": "numeric"},
        "ORDERDATE": {"dtype": "date"},
        "QUANTITY": {"dtype": "numeric"},
        "COSTCENTER": {"dtype": "string"},
        "DISCOUNT_RATE": {"dtype": "numeric"},
    },
    "rows": table.to_dict(orient="records"),
}

client = RPTClientAPI(model_name="sap-rpt-1.6")
response = client.predict_json(payload)
print("status:", response["status"])
print("metadata:", response["metadata"])

status: {'code': 0, 'message': 'ok'}
metadata: {'context_mode': 'default', 'num_columns': 7, 'num_predictions': 4, 'num_query_rows': 2, 'num_rows': 10}


## 5. Read the predictions

The response has one record per query row, and inside it one list of answers per target column. `flatten_predictions` turns that into a table with one row per answer:

* **Category (`COSTCENTER`)**: one row per alternative (`rank` 1 = best). `confidence` is the probability the model gives to that value: 0.86 means "86% likely". The alternatives of one cell add up to about 1.
* **Number (`DISCOUNT_RATE`)**: one row with the estimate in `prediction` and a likely range `[low, high]` in `confidence_interval`. `confidence` is empty, because a probability per value does not exist for numbers; the range expresses the uncertainty instead.

In [14]:
predictions = pd.DataFrame(flatten_predictions(response, index_column="ID"))
predictions["actual"] = [truth.loc[i, t] for i, t in zip(predictions["index"], predictions["target"])]
predictions

,query_position,index,target,rank,prediction,confidence,confidence_interval,actual
0,0,35,COSTCENTER,1,Office Furniture,0.86,None,Office Furniture
1,0,35,COSTCENTER,2,Data Infrastructure,0.14,None,Office Furniture
2,0,35,DISCOUNT_RATE,1,0.11661,NaN,"[0.0994, 0.1437]",0.09
3,1,711,COSTCENTER,1,Data Infrastructure,0.74,None,Data Infrastructure
4,1,711,COSTCENTER,2,Office Furniture,0.26,None,Data Infrastructure
5,1,711,DISCOUNT_RATE,1,0.092333,NaN,"[0.0505, 0.105]",0.11


## 6. Explanations

When `explanations` is requested, the response also says, for each query row:

* **`top_column_scores`**: the columns the model paid most attention to, with a weight. In the output below, `PRICE` and `PRODUCT` mattered most. The weights show where the model looked, not cause and effect, and a shortened top-3 list does not add up to 1, so do not read them as percentages.
* **`top_relevant_context_rows`**: the example rows the model found most relevant, as positions in the `rows` list you sent (0 = first row). `map_explanation_rows` looks those rows up for you.

In a request with several targets there is one explanation per query row, shared by all targets. Each count can be 1 to 20.

In [15]:
for query_id, explanation in zip(QUERY_IDS, map_explanation_rows(response, payload["rows"])):
    print(f"Order {query_id}")
    print("  columns that mattered most:", explanation["column_scores"])
    for context in explanation["relevant_context_rows"]:
        row = context["row"]
        print(f"  most relevant example (row {context['position']}): {row['PRODUCT']} -> {row['COSTCENTER']}, {row['DISCOUNT_RATE']}")

Order 35
  columns that mattered most: {'PRICE': 0.382, 'PRODUCT': 0.3, 'QUANTITY': 0.214}
  most relevant example (row 5): Network Switch -> Data Infrastructure, 0.12
  most relevant example (row 3): Standing Desk -> Office Furniture, 0.1
Order 711
  columns that mattered most: {'PRICE': 0.382, 'PRODUCT': 0.328, 'QUANTITY': 0.198}
  most relevant example (row 5): Network Switch -> Data Infrastructure, 0.12
  most relevant example (row 1): Server Rack -> Data Infrastructure, 0.15


## 7. Comparing the model variants on real data

Ten orders are enough to see how a call works, but not to judge quality: two answers can be right or wrong by luck. A fair comparison needs many rows **whose answers we know but hide from the model** (a *holdout*), and a simple baseline to beat.

**Data.** `data/employee_sample.csv` is a 2,500-row sample of the employee dataset from the SAP HANA Cloud Machine Learning Challenge ("I quit! How to predict employee churn"), also used by this repository's classification best practice. Each row is one employee with age, tenure, job level, functional area, country, performance ratings and promotion history, plus:

* `SALARY`: annual salary (a number, in 11 bands from 20,000 to 120,000), our **regression** target;
* `FLIGHT_RISK`: whether the employee was flagged as likely to leave (`Yes`/`No`), our **classification** target. Only about 1 in 10 employees is `Yes`.

**Setup.** 2,000 rows are context (answers visible), 500 are queries (answers hidden). Both targets are predicted in one request per model. The small models accept up to 2,048 context rows plus 512 query rows (2,560 rows in total).

**Tip on IDs.** The service reads values that look like numbers as numbers, so an ID such as `29089` comes back as a number and no longer matches your text key. Make IDs clearly text, e.g. `emp-29089`.

In [16]:
employees = pd.read_csv("data/employee_sample.csv")
employees["EMPLOYEE_ID"] = "emp-" + employees["EMPLOYEE_ID"].astype(str)
# JSON has no NaN: send empty cells as null.
employees = employees.astype(object).where(employees.notna(), None)
context_rows, query_rows = employees.iloc[500:], employees.iloc[:500]
actual_salary = query_rows["SALARY"].astype(float).to_numpy()
actual_risk = query_rows["FLIGHT_RISK"].to_numpy()
print(f"{len(context_rows)} context rows, {len(query_rows)} query rows, {employees.shape[1]} columns")
print(f"share of FLIGHT_RISK = Yes among the queries: {np.mean(actual_risk == 'Yes'):.1%}")

2000 context rows, 500 query rows, 42 columns
share of FLIGHT_RISK = Yes among the queries: 9.6%


In [17]:
hidden = query_rows.copy()
hidden[["SALARY", "FLIGHT_RISK"]] = "[PREDICT]"
holdout_payload = {
    "index_column": "EMPLOYEE_ID",
    "prediction_config": {
        "target_columns": [
            {"name": "SALARY", "prediction_placeholder": "[PREDICT]", "task_type": "regression"},
            {"name": "FLIGHT_RISK", "prediction_placeholder": "[PREDICT]", "task_type": "classification", "top_k": 2},
        ]
    },
    "rows": context_rows.to_dict(orient="records") + hidden.to_dict(orient="records"),
}

MODELS = ["sap-rpt-1.5", "sap-rpt-1.5-large", "sap-rpt-1.6", "sap-rpt-1.6-large"]
holdout = {}
for model in tqdm(MODELS, desc="models"):
    started = time.perf_counter()
    model_response = RPTClientAPI(model_name=model).predict_json(holdout_payload, compress=True)
    by_id = {record["EMPLOYEE_ID"]: record for record in model_response["predictions"]}
    records = [by_id[i] for i in query_rows["EMPLOYEE_ID"]]
    holdout[model] = {
        "salary": np.array([r["SALARY"][0]["prediction"] for r in records]),
        "salary_range": np.array([r["SALARY"][0]["confidence_interval"] for r in records]),
        "p_yes": np.array([next((c["confidence"] for c in r["FLIGHT_RISK"] if c["prediction"] == "Yes"), 0.0)
                           for r in records]),
        "seconds": time.perf_counter() - started,
    }

models: 100%|██████████| 4/4 [00:10<00:00,  2.60s/it]


**How to read the results.**

* **Salary error**: the average distance between estimated and real salary. Baseline: guess the median salary of the context for everyone.
* **Salary range hit rate**: how often the real salary lies inside the returned range.
* **Flight risk, accuracy**: share of correct Yes/No answers. Careful: because only ~10% are `Yes`, always answering `No` is already about 90% accurate, so accuracy alone says little.
* **Flight risk, top-50 hit rate**: sort the 500 employees by the model's probability of `Yes` and look at the 50 highest. Which share of them really is `Yes`? This is how such a model is used in practice (who should HR talk to first?). Picking 50 at random would hit about 10%.

In [18]:
baseline_salary = float(np.median(context_rows["SALARY"].astype(float)))
rows = {"baseline": {
    "salary error": np.abs(baseline_salary - actual_salary).mean(),
    "flight risk accuracy": np.mean(actual_risk == "No"),
    "flight risk top-50 hit rate": np.mean(actual_risk == "Yes"),
}}
for model, result in holdout.items():
    lower, upper = result["salary_range"][:, 0], result["salary_range"][:, 1]
    top_50 = np.argsort(-result["p_yes"])[:50]
    rows[model] = {
        "salary error": np.abs(result["salary"] - actual_salary).mean(),
        "salary range hit rate": np.mean((actual_salary >= lower) & (actual_salary <= upper)),
        "flight risk accuracy": np.mean(np.where(result["p_yes"] >= 0.5, "Yes", "No") == actual_risk),
        "flight risk top-50 hit rate": np.mean(actual_risk[top_50] == "Yes"),
        "latency (s)": result["seconds"],
    }
pd.DataFrame(rows).T.round(3)

,salary error,flight risk accuracy,flight risk top-50 hit rate,salary range hit rate,latency (s)
baseline,18248.000,0.904,0.096,NaN,NaN
sap-rpt-1.5,8811.442,0.912,0.400,0.984,4.103
sap-rpt-1.5-large,8811.442,0.912,0.400,0.984,2.110
sap-rpt-1.6,8163.547,0.914,0.420,0.978,2.091
sap-rpt-1.6-large,8163.547,0.914,0.420,0.978,2.090


What the table shows in our run:

* Both model generations clearly beat the baselines: the salary error is less than half of the baseline's, and the 50 employees ranked riskiest contain about four times more real `Yes` cases than a random pick.
* SAP-RPT-1.6 is somewhat more accurate than SAP-RPT-1.5 on both targets.
* The large variants gave the same answers as the small ones here. With 2,000 context rows both fit everything; the large variants are built for much more context (up to 65,536 rows), see `02_RPT-1.6_deep_context.ipynb`.
* The returned salary ranges contained the real salary in almost all cases, so they are on the generous side for this data.

Model limits (SAP Help, SAP-RPT-1.5 and SAP-RPT-1.6 model pages):

| Model | Max context rows | Recommended context | Max columns | Use |
|---|---|---|---|---|
| `sap-rpt-1.5` | 2,048 | 500-2,000 | 100 | previous generation, explanations |
| `sap-rpt-1.5-large` | 65,536 | 4,000-8,000 | 256 | previous generation, more context |
| `sap-rpt-1.6` | 2,048 | 500-2,000 | 100 | default, low latency |
| `sap-rpt-1.6-large` | 65,536 | 4,000-36,000 | 256 | large context, many categories, `deep` mode |

All variants accept up to 512 query rows and 10 target columns per request.

## 8. Larger requests: compression

For large tables, `predict_json(payload, compress=True)` compresses the request body (gzip, header `Content-Encoding: gzip`), as done in the comparison above. The answers are identical:

In [19]:
compressed = client.predict_json(payload, compress=True)
print("same predictions:", compressed["predictions"] == response["predictions"])

same predictions: True


## Summary

* An RPT call is `POST {deployment_url}/predict` with a table in which the unknown cells are marked `[PREDICT]`. Check both the HTTP status and `status.code`.
* Set `task_type`, `top_k` and `data_schema` explicitly, keep a text `index_column` to match answers to rows.
* Categories come with probabilities per alternative; numbers come with a likely range.
* Request `explanations` only when you use them, and look up the returned row positions in the table you sent.
* Judge quality on a holdout of many rows with known answers and a simple baseline, never on a handful of examples.
* Next: `02_RPT-1.6_deep_context.ipynb` for large context with `sap-rpt-1.6-large`.